<a href="https://colab.research.google.com/github/Sricharan-rec/Gen-AI_Lab-AD23633/blob/main/genAI_Exp_5_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run once at the start of your Colab session
# The ! prefix tells Colab to run this as a shell command
!pip install "transformers==4.35.0"
# PDF reading — extracts plain text from .pdf files
from pypdf import PdfReader
# Embedding model — converts text chunks into 384-dim float vectors
# Similar texts produce nearby vectors (enables meaning-based search)
from sentence_transformers import SentenceTransformer
# Vector database — stores and searches embedding vectors by similarity
import faiss
import numpy as np # FAISS requires float32 numpy arrays
# Local LLM — flan-t5-base runs in Colab without any API key
# text2text-generation: takes a prompt string, returns an answer string
from transformers import pipeline
# Standard library
from typing import List # safer than list[str] for Python < 3.10
from textwrap import wrap as word_wrap

def load_pdf(path: str) -> str:
 # PdfReader opens the file and gives us one Page object per page
 # extract_text() returns the plain text on that page
 # Some pages may be image-only — extract_text() returns '' safely
 reader = PdfReader(path)
 full_text = ""
 for page in reader.pages:
  full_text += page.extract_text() or ""
 return full_text

def chunk_text(text: str,
 chunk_size: int = 500,
 overlap: int = 50) -> List[str]:
 # chunk_size = 500 words ~= 700 tokens (fits safely in most LLMs)
 # overlap = 50 words — last 50 words of chunk N reappear at
 # the start of chunk N+1, so no context is lost
 # at a chunk boundary
 words = text.split() # split on whitespace
 chunks = []
 start = 0
 while start < len(words):
  end = start + chunk_size
  chunk = " ".join(words[start:end])
  chunks.append(chunk)
  start += chunk_size - overlap # slide window forward
 return chunks

def build_embeddings(chunks: List[str],
 model_name: str = "all-MiniLM-L6-v2"):
 # Load the embedding model (downloaded once, ~80 MB, runs locally)
 # No internet or API key needed after the first download
 print(f" Loading embedding model: {model_name} ...")
 embedder = SentenceTransformer(model_name)
 # encode() processes all chunks and returns a numpy array
 # of shape (N_chunks, 384) — each row is one chunk's vector
 print(f" Embedding {len(chunks)} chunks ...")
 embeddings = embedder.encode(chunks, show_progress_bar=True)
 # FAISS requires float32 — we cast explicitly to be safe
 return embedder, embeddings.astype("float32")

def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
 # dimension must match the embedding model output
 # all-MiniLM-L6-v2 produces 384-dimensional vectors
 dimension = embeddings.shape[1]
 # IndexFlatL2: exact brute-force nearest-neighbour search
 # using L2 (Euclidean) distance — lower = more similar
 index = faiss.IndexFlatL2(dimension)
 # add() copies all chunk vectors into the index
 # After this, index.ntotal == number of chunks stored
 index.add(embeddings)
 print(f" FAISS index built: {index.ntotal} vectors stored.")
 return index

def retrieve(query: str,
 embedder,
 index: faiss.Index,
 chunks: List[str],
 top_k: int = 3) -> List[str]:
 # Step 1: embed the question into the same 384-D vector space
 # We MUST use the same model used to embed the chunks
 # Shape: (1, 384) — one query, 384 dimensions
 query_vec = embedder.encode([query]).astype("float32")
 # Step 2: search the FAISS index for the top_k nearest vectors
 # distances shape: (1, top_k) — L2 distances (lower = more similar)
 # indices shape: (1, top_k) — positions in our chunks list
 distances, indices = index.search(query_vec, top_k)
 # Step 3: map indices back to actual text chunks
 # indices[0] = results for our single query
 # Guard against -1 (means FAISS found no result in that slot)
 retrieved = [chunks[idx] for idx in indices[0] if idx != -1]
 return retrieved

def generate_answer(query: str,
 context_chunks: List[str],
 generator,
 max_tokens: int = 300) -> str:
 # Join retrieved chunks with a clear separator
 # so the LLM can see where one passage ends and the next begins
 context = "\n\n---\n\n".join(context_chunks)
 # Prompt structure:
 # [Instruction] — tells the model to stay grounded in context
 # [Context] — the retrieved document passages
 # [Question] — the user's query
 # Grounding prevents the model from hallucinating facts
 prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say
\"I don't know based on the provided documents.\"
Context:
{context}
Question: {query}
Answer:"""
 # generator() runs flan-t5-base locally — no internet call
 # max_new_tokens controls the maximum length of the response
 # Result format: [{'generated_text': '...'}]
 result = generator(prompt, max_new_tokens=max_tokens)
 return result[0]["generated_text"].strip()

def build_rag_pipeline(pdf_paths: List[str]):
 # Load, chunk, and collect text from every PDF
 all_chunks = []
 for pdf_path in pdf_paths:
  print(f"\n[+] Loading: {pdf_path}")
  text = load_pdf(pdf_path)
  chunks = chunk_text(text, chunk_size=500, overlap=50)
  print(f" -> {len(chunks)} chunks extracted")
  all_chunks.extend(chunks)
 print(f"\n[+] Total chunks: {len(all_chunks)}")
 # Embed all chunks into 384-D vectors
 print("\n[+] Building embeddings ...")
 embedder, embeddings = build_embeddings(all_chunks)
 # Store all vectors in FAISS
 print("\n[+] Building FAISS index ...")
 index = build_faiss_index(embeddings)
 return embedder, index, all_chunks

def ask(query: str, embedder, index, chunks, generator, top_k: int = 3):
 print(f"\n{'='*60}")
 print(f"QUESTION: {query}")
 print(f"{'='*60}")
 # Step 4: Retrieve the most relevant chunks
 print("\n[Retrieve] Searching FAISS index ...")
 context = retrieve(query, embedder, index, chunks, top_k=top_k)
 print(f" -> {len(context)} chunks retrieved")
 # Step 5: Generate a grounded answer
 print("\n[Generate] Calling flan-t5-base ...")
 answer = generate_answer(query, context, generator)
 print("\nANSWER:\n")
 for line in word_wrap(answer, width=70):
  print(" ", line)
 return answer

# 1. Point to your uploaded PDFs
# Upload files via Colab sidebar (folder icon) before running this
PDF_FILES = [
 "paper1_attention_is_all_you_need.pdf", # change to your filename
 "paper2_bert_pretraining.pdf", # change to your filename
]

# 2. Build the index (run once, reuse for many questions)
embedder, index, chunks = build_rag_pipeline(PDF_FILES)

# 3. Load the local LLM
# flan-t5-base: ~250 MB, downloads automatically, no API key
# Upgrade option: "google/flan-t5-large" (~800 MB, better answers)
print("\n[+] Loading local LLM: flan-t5-base ...")
generator = pipeline(
 "text2text-generation",
 model="google/flan-t5-base"
)

# 4. Ask questions
questions = [
 "What is the core idea behind the attention mechanism?"
]

for q in questions:
 ask(q, embedder, index, chunks, generator)


[+] Loading: paper1_attention_is_all_you_need.pdf
 -> 14 chunks extracted

[+] Loading: paper2_bert_pretraining.pdf
 -> 23 chunks extracted

[+] Total chunks: 37

[+] Building embeddings ...
 Loading embedding model: all-MiniLM-L6-v2 ...
 Embedding 37 chunks ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


[+] Building FAISS index ...
 FAISS index built: 37 vectors stored.

[+] Loading local LLM: flan-t5-base ...


Token indices sequence length is longer than the specified maximum sequence length for this model (2371 > 512). Running this sequence through the model will result in indexing errors



QUESTION: What is the core idea behind the attention mechanism?

[Retrieve] Searching FAISS index ...
 -> 3 chunks retrieved

[Generate] Calling flan-t5-base ...

ANSWER:

  Multi-head attention
